In [1]:
!pip install -q pillow==11.2.1
!pip install -q transformers==4.57.1 peft==0.18.0 accelerate==1.11.0 bitsandbytes==0.48.1 sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 88.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 142.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 71.9 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os, gc, ast, json, time, random, math, re
import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, AutoModelForVision2Seq
from peft import LoraConfig, get_peft_model, PeftModel, TaskType

Mounted at /content/drive


In [3]:
ROOT = Path("/content/drive/MyDrive/pixels-to-predictions")
TRAIN_CSV = ROOT / "train.csv"
VAL_CSV   = ROOT / "val.csv"
TEST_CSV  = ROOT / "test.csv"
IMG_ROOT = ROOT / "images" / "images"

BASE_MODEL_NAME = "HuggingFaceTB/SmolVLM-500M-Instruct"

OUT_DIR = ROOT / "outputs_smolvlm_nli_under5m"
BEST_DIR = OUT_DIR / "best_adapter"
LAST_DIR = OUT_DIR / "last_adapter"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SUB_PATH = ROOT / "submission_nli.csv"
DEBUG_PATH = ROOT / "submission_nli_debug.csv"

LETTERS = ["A", "B", "C", "D", "E"]
SEED = 42
MAX_IMAGE_SIZE = 448

EPOCHS = 3
BATCH_SIZE = 1
GRAD_ACCUM = 8
LR = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 1.0
NUM_WORKERS = 2
PRINT_EVERY = 50

USE_IMAGE_PREPROCESSING = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

assert TRAIN_CSV.exists(), f"Missing {TRAIN_CSV}"
assert VAL_CSV.exists(), f"Missing {VAL_CSV}"
assert TEST_CSV.exists(), f"Missing {TEST_CSV}"

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

train: (3109, 15)
val: (1048, 15)
test: (1008, 13)


In [4]:
def parse_choices(choices):
    if choices is None:
        return []
    if isinstance(choices, list):
        return choices
    if isinstance(choices, dict):
        return [choices[k] for k in LETTERS if k in choices]
    s = str(choices).strip()
    if s.lower() in ["", "nan", "none"]:
        return []
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, list):
            return parsed
        if isinstance(parsed, dict):
            return [parsed[k] for k in LETTERS if k in parsed]
    except Exception:
        pass
    try:
        parsed = json.loads(s)
        if isinstance(parsed, list):
            return parsed
        if isinstance(parsed, dict):
            return [parsed[k] for k in LETTERS if k in parsed]
    except Exception:
        pass
    return [s]


def find_image(row, split):
    raw = str(row.get("image_path", "")).strip()
    qid = str(row.get("id", "")).strip()
    candidates = []

    if raw and raw.lower() not in ["", "nan", "none"]:
        raw_path = Path(raw)
        candidates.extend([
            ROOT / raw,
            ROOT / "images" / raw_path.name,
            ROOT / "images" / split / raw_path.name,
            ROOT / "images" / "images" / raw_path.name,
            ROOT / "images" / "images" / split / raw_path.name,
            IMG_ROOT / split / raw_path.name,
        ])

    if qid and qid.lower() not in ["", "nan", "none"]:
        for ext in [".png", ".jpg", ".jpeg", ".webp"]:
            candidates.extend([
                IMG_ROOT / split / f"{qid}{ext}",
                ROOT / "images" / "images" / split / f"{qid}{ext}",
                ROOT / "images" / split / f"{qid}{ext}",
            ])

    for c in candidates:
        if c.exists():
            return c
    return None


def is_image_heavy(row):
    text = " ".join([
        str(row.get("question", "")), str(row.get("skill", "")),
        str(row.get("topic", "")), str(row.get("category", "")),
        str(row.get("hint", "")), str(row.get("lecture", "")),
    ]).lower()
    keys = [
        "image", "picture", "diagram", "graph", "chart", "table", "map", "model",
        "shown", "figure", "punnett", "square", "label", "labels", "axis", "axes",
        "food web", "circuit", "rock cycle", "life cycle", "map", "drawing"
    ]
    return any(k in text for k in keys)


def preprocess_image(img, row=None):
    img = img.convert("RGB")
    w, h = img.size
    scale = min(MAX_IMAGE_SIZE / max(w, h), 1.0)
    if scale < 1.0:
        img = img.resize((int(w * scale), int(h * scale)), Image.Resampling.BICUBIC)
    if USE_IMAGE_PREPROCESSING and row is not None and is_image_heavy(row):
        img = ImageEnhance.Contrast(img).enhance(1.45)
        img = ImageEnhance.Sharpness(img).enhance(1.7)
    return img


def load_image(row, split):
    img_path = find_image(row, split)
    if img_path is None:
        return Image.new("RGB", (MAX_IMAGE_SIZE, MAX_IMAGE_SIZE), "white"), None
    try:
        img = Image.open(img_path).convert("RGB")
        img = preprocess_image(img, row=row)
        return img, img_path
    except Exception as e:
        print("Image load failed:", img_path, repr(e))
        return Image.new("RGB", (MAX_IMAGE_SIZE, MAX_IMAGE_SIZE), "white"), None


def detect_question_type(row):
    text = " ".join([
        str(row.get("question", "")), str(row.get("skill", "")),
        str(row.get("topic", "")), str(row.get("category", "")),
        str(row.get("hint", "")), str(row.get("lecture", "")),
    ]).lower()
    if "punnett" in text or "homozygous" in text or "heterozygous" in text or "offspring" in text:
        return "punnett"
    if "graph" in text or "chart" in text or "axis" in text or "axes" in text:
        return "graph"
    if "table" in text:
        return "table"
    if "map" in text:
        return "map"
    if "food web" in text or "food chain" in text:
        return "food_web"
    if "circuit" in text or "electric" in text:
        return "circuit"
    if "passage" in text or "read the text" in text or "informational text" in text or "reading-comprehension" in text:
        return "passage"
    if "diagram" in text or "model" in text or "shown" in text or "figure" in text:
        return "diagram"
    return "general"


def type_guidance(qtype):
    guides = {
        "punnett": "Punnett/genetics: inspect the four boxes. Count homozygous dominant, homozygous recessive, heterozygous, dominant phenotype, or recessive phenotype depending on the exact question. For ratios, preserve the order asked.",
        "graph": "Graph/chart: read title, axes, units, labels, legend, and compare values/trends carefully.",
        "table": "Table: read row labels, column labels, and exact cell values. Match the relevant row/column to the question.",
        "map": "Map: use labels, legend, direction, regions, and relative location.",
        "food_web": "Food web/chain: follow arrow direction carefully. Energy usually flows from food to consumer. Identify producer, consumer, predator, and prey.",
        "circuit": "Circuit: trace the path, decide open/closed, and identify battery, bulb, switch, wire, conductor, or insulator.",
        "passage": "Reading passage: use evidence from the passage/hint. Make only supported inferences.",
        "diagram": "Diagram/model: read labels/arrows/parts and use the visual relationship shown.",
        "general": "General science: use question, choices, lecture, hint, metadata, and image if useful."
    }
    return guides.get(qtype, guides["general"])


def build_nli_prompt(row, choice_text):
    question = str(row.get("question", "")).strip()
    subject = str(row.get("subject", "")).strip()
    topic = str(row.get("topic", "")).strip()
    category = str(row.get("category", "")).strip()
    skill = str(row.get("skill", "")).strip()
    lecture = str(row.get("lecture", "")).strip()
    hint = str(row.get("hint", "")).strip()
    qtype = detect_question_type(row)
    guide = type_guidance(qtype)

    return f"""
You are judging whether a proposed answer choice is correct for a science multiple-choice question.

Use only the provided image and context. Decide whether the proposed answer is supported.

Question type guidance:
{guide}

Question:
{question}

Subject: {subject}
Topic: {topic}
Category: {category}
Skill: {skill}

Lecture:
{lecture}

Hint:
{hint}

Proposed answer:
{choice_text}

Is the proposed answer correct?
Answer only Yes or No.
""".strip()

In [5]:
processor = AutoProcessor.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
base_model = AutoModelForVision2Seq.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(base_model, lora_config)
model.train()

def count_trainable_params(model):
    trainable = 0
    total = 0
    for p in model.parameters():
        total += p.numel()
        if p.requires_grad:
            trainable += p.numel()
    return trainable, total

trainable, total = count_trainable_params(model)
print(f"Trainable params: {trainable:,}")
print(f"Total params: {total:,}")
print(f"Trainable %: {100 * trainable / total:.4f}%")
assert trainable <= 5_000_000, f"Too many trainable params: {trainable:,}"
model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Trainable params: 4,784,128
Total params: 512,266,432
Trainable %: 0.9339%
trainable params: 4,784,128 || all params: 512,266,432 || trainable%: 0.9339


In [6]:
def get_token_ids_for_words(words):
    ids = []
    for w in words:
        toks = processor.tokenizer(w, add_special_tokens=False)["input_ids"]
        if toks:
            ids.append(toks[-1])
    return list(set(ids))

YES_TOKEN_IDS = get_token_ids_for_words(["Yes", " yes", "Yes.", " yes.", "YES"])
NO_TOKEN_IDS  = get_token_ids_for_words(["No", " no", "No.", " no.", "NO"])
print("YES_TOKEN_IDS:", YES_TOKEN_IDS)
print("NO_TOKEN_IDS:", NO_TOKEN_IDS)
assert len(YES_TOKEN_IDS) > 0

def make_chat_text(prompt):
    clean_prompt = str(prompt).replace("<image>", "").strip()
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": clean_prompt}]}]
    return processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

class NLIMCQDataset(Dataset):
    def __init__(self, df, split):
        self.df = df.reset_index(drop=True)
        self.split = split

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image, img_path = load_image(row, self.split)
        choices = parse_choices(row.get("choices", ""))
        answer_idx = int(row["answer"])
        prompts = []
        for i, choice in enumerate(choices):
            letter = LETTERS[i]
            choice_text = f"{letter}. {str(choice).strip()}"
            prompts.append(build_nli_prompt(row, choice_text))
        return {
            "row_idx": idx,
            "id": row.get("id", idx),
            "image": image,
            "prompts": prompts,
            "answer_idx": answer_idx,
            "num_choices": len(choices),
            "is_image_heavy": is_image_heavy(row),
        }


def collate_nli(batch):
    all_texts, all_images, group_sizes, labels = [], [], [], []
    for item in batch:
        prompts = item["prompts"]
        image = item["image"]
        group_sizes.append(len(prompts))
        labels.append(item["answer_idx"])
        for prompt in prompts:
            all_texts.append(make_chat_text(prompt))
            all_images.append(image)
    inputs = processor(text=all_texts, images=all_images, return_tensors="pt", padding=True, truncation=True)
    return {"inputs": inputs, "group_sizes": group_sizes, "labels": torch.tensor(labels, dtype=torch.long)}

train_ds = NLIMCQDataset(train_df, "train")
val_ds = NLIMCQDataset(val_df, "val")
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collate_nli, pin_memory=True)

YES_TOKEN_IDS: [2097, 10539, 9805, 30]
NO_TOKEN_IDS: [13465, 787, 5230, 30]


In [7]:
def get_yes_score_from_logits(logits):
    # logits: [num_choices, vocab]
    yes_logits = logits[:, YES_TOKEN_IDS]
    return yes_logits.max(dim=1).values


def nli_forward_loss(batch):
    inputs = batch["inputs"]
    labels = batch["labels"]
    model_device = next(model.parameters()).device
    inputs = {k: v.to(model_device) if torch.is_tensor(v) else v for k, v in inputs.items()}
    labels = labels.to(model_device)

    outputs = model(**inputs)
    logits = outputs.logits[:, -1, :]

    losses = []
    start = 0
    for b, group_size in enumerate(batch["group_sizes"]):
        end = start + group_size
        group_logits = logits[start:end]
        yes_scores = get_yes_score_from_logits(group_logits).unsqueeze(0)
        correct_idx = labels[b].unsqueeze(0)
        loss = torch.nn.functional.cross_entropy(yes_scores.float(), correct_idx)
        losses.append(loss)
        start = end
    return torch.stack(losses).mean()

@torch.no_grad()
def nli_score_one(row, split="val", use_blank_image=False):
    model.eval()
    image, img_path = load_image(row, split)
    if use_blank_image:
        image = Image.new("RGB", image.size, "white")
    choices = parse_choices(row.get("choices", ""))
    prompts = []
    for i, choice in enumerate(choices):
        prompts.append(build_nli_prompt(row, f"{LETTERS[i]}. {str(choice).strip()}"))
    texts = [make_chat_text(p) for p in prompts]
    images = [image for _ in prompts]
    inputs = processor(text=texts, images=images, return_tensors="pt", padding=True, truncation=True)
    model_device = next(model.parameters()).device
    inputs = {k: v.to(model_device) if torch.is_tensor(v) else v for k, v in inputs.items()}
    outputs = model(**inputs)
    logits = outputs.logits[:, -1, :]
    yes_scores = get_yes_score_from_logits(logits)
    pred = int(torch.argmax(yes_scores).item())
    return pred, yes_scores.detach().cpu().tolist(), img_path

@torch.no_grad()
def evaluate_val_nli(use_blank_image=False, max_rows=None):
    model.eval()
    df = val_df.iloc[:max_rows] if max_rows is not None else val_df
    correct = total_eval = 0
    image_correct = image_total = 0
    text_correct = text_total = 0
    for _, row in tqdm(df.iterrows(), total=len(df), desc="NLI val blank" if use_blank_image else "NLI val real"):
        y = int(row["answer"])
        pred, _, _ = nli_score_one(row, split="val", use_blank_image=use_blank_image)
        ok = int(pred == y)
        correct += ok
        total_eval += 1
        if is_image_heavy(row):
            image_correct += ok
            image_total += 1
        else:
            text_correct += ok
            text_total += 1
    model.train()
    return {
        "acc": correct / max(total_eval, 1),
        "image_heavy_acc": image_correct / max(image_total, 1),
        "text_acc": text_correct / max(text_total, 1),
    }

In [8]:
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=WEIGHT_DECAY)
total_update_steps = math.ceil(len(train_loader) / GRAD_ACCUM) * EPOCHS
warmup_steps = int(total_update_steps * WARMUP_RATIO)

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_update_steps - warmup_steps)
    return max(0.05, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

In [9]:
best_acc = -1.0
history = []
global_step = 0

print("Starting NLI-style training...")
print("Total update steps:", total_update_steps)
print("Warmup steps:", warmup_steps)

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_start = time.time()
    running_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(train_loader, start=1):
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            loss = nli_forward_loss(batch) / GRAD_ACCUM
        scaler.scale(loss).backward()
        running_loss += loss.item() * GRAD_ACCUM

        if step % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1

        if step % PRINT_EVERY == 0:
            elapsed = (time.time() - epoch_start) / 60
            steps_per_min = step / max(elapsed, 1e-6)
            eta = (len(train_loader) - step) / max(steps_per_min, 1e-6)
            avg_loss = running_loss / step
            print(f"Epoch {epoch}/{EPOCHS} | step {step}/{len(train_loader)} | loss {avg_loss:.4f} | lr {scheduler.get_last_lr()[0]:.2e} | elapsed {elapsed:.1f}m | ETA {eta:.1f}m")

    epoch_min = (time.time() - epoch_start) / 60
    avg_loss = running_loss / max(len(train_loader), 1)
    print(f"\nEpoch {epoch} finished in {epoch_min:.2f} min | avg loss {avg_loss:.4f}")

    print("Running NLI validation with REAL images...")
    real = evaluate_val_nli(use_blank_image=False)
    print("Running NLI validation with BLANK images...")
    blank = evaluate_val_nli(use_blank_image=True)

    row_hist = {
        "epoch": epoch,
        "epoch_min": epoch_min,
        "loss": avg_loss,
        "real_acc": real["acc"],
        "real_image_heavy_acc": real["image_heavy_acc"],
        "real_text_acc": real["text_acc"],
        "blank_acc": blank["acc"],
        "blank_image_heavy_acc": blank["image_heavy_acc"],
        "blank_text_acc": blank["text_acc"],
    }
    history.append(row_hist)
    print("\nEPOCH SUMMARY")
    for k, v in row_hist.items():
        print(k, ":", v)

    pd.DataFrame(history).to_csv(OUT_DIR / "training_history.csv", index=False)
    model.save_pretrained(LAST_DIR)
    processor.save_pretrained(LAST_DIR)

    if real["acc"] > best_acc:
        best_acc = real["acc"]
        model.save_pretrained(BEST_DIR)
        processor.save_pretrained(BEST_DIR)
        print("Saved new BEST adapter:", BEST_DIR, "acc:", best_acc)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nTRAINING DONE")
print("Best real-image val acc:", best_acc)
print("Best adapter:", BEST_DIR)
display(pd.DataFrame(history))

Starting NLI-style training...
Total update steps: 1167
Warmup steps: 58


/tmp/ipykernel_2820/467436258.py:27: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


Epoch 1/3 | step 50/3109 | loss 7.6754 | lr 2.07e-05 | elapsed 0.7m | ETA 40.1m
Epoch 1/3 | step 100/3109 | loss 6.7634 | lr 4.14e-05 | elapsed 1.1m | ETA 32.9m
Epoch 1/3 | step 150/3109 | loss 6.3009 | lr 6.21e-05 | elapsed 1.5m | ETA 29.5m
Epoch 1/3 | step 200/3109 | loss 5.4449 | lr 8.62e-05 | elapsed 1.9m | ETA 28.0m
Epoch 1/3 | step 250/3109 | loss 4.5642 | lr 1.07e-04 | elapsed 2.3m | ETA 26.7m
Epoch 1/3 | step 300/3109 | loss 3.9602 | lr 1.28e-04 | elapsed 2.7m | ETA 25.6m
Epoch 1/3 | step 350/3109 | loss 3.5284 | lr 1.48e-04 | elapsed 3.2m | ETA 25.1m
Epoch 1/3 | step 400/3109 | loss 3.2450 | lr 1.72e-04 | elapsed 3.6m | ETA 24.5m
Epoch 1/3 | step 450/3109 | loss 2.9897 | lr 1.93e-04 | elapsed 4.0m | ETA 23.8m
Epoch 1/3 | step 500/3109 | loss 2.7756 | lr 2.00e-04 | elapsed 4.4m | ETA 23.2m
Epoch 1/3 | step 550/3109 | loss 2.6004 | lr 2.00e-04 | elapsed 4.8m | ETA 22.6m
Epoch 1/3 | step 600/3109 | loss 2.4554 | lr 2.00e-04 | elapsed 5.3m | ETA 22.1m
Epoch 1/3 | step 650/3109 | l

NLI val real:   0%|          | 0/1048 [00:00<?, ?it/s]

Running NLI validation with BLANK images...


...

NLI val real:   0%|          | 0/1048 [00:00<?, ?it/s]

Running NLI validation with BLANK images...


NLI val blank:   0%|          | 0/1048 [00:00<?, ?it/s]


EPOCH SUMMARY
epoch : 3
epoch_min : 17.436090664068857
loss : 0.1674423141996689
real_acc : 0.8368320610687023
real_image_heavy_acc : 0.8401360544217688
real_text_acc : 0.8192771084337349
blank_acc : 0.6669847328244275
blank_image_heavy_acc : 0.6496598639455783
blank_text_acc : 0.7590361445783133
Saved new BEST adapter: /content/drive/MyDrive/pixels-to-predictions/outputs_smolvlm_nli_under5m/best_adapter acc: 0.8368320610687023

TRAINING DONE
Best real-image val acc: 0.8368320610687023
Best adapter: /content/drive/MyDrive/pixels-to-predictions/outputs_smolvlm_nli_under5m/best_adapter


,epoch,epoch_min,loss,real_acc,real_image_heavy_acc,real_text_acc,blank_acc,blank_image_heavy_acc,blank_text_acc
0,1,25.466440,1.028785,0.720420,0.705215,0.801205,0.620229,0.592971,0.765060
1,2,17.438742,0.360600,0.822519,0.820862,0.831325,0.664122,0.642857,0.777108
2,3,17.436091,0.167442,0.836832,0.840136,0.819277,0.666985,0.649660,0.759036


In [10]:
print("Reloading best adapter for test inference...")
del model
try:
    del base_model
except Exception:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

base_model = AutoModelForVision2Seq.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(base_model, BEST_DIR, is_trainable=False)
model.eval()

pred_rows = []
t0 = time.time()

for idx, row in test_df.iterrows():
    try:
        pred, scores, img_path = nli_score_one(row, split="test", use_blank_image=False)
        pred_rows.append({
            "id": row["id"],
            "answer": int(pred),
            "scores": scores,
            "image_path": str(img_path) if img_path else "",
            "error": "",
        })
    except Exception as e:
        pred_rows.append({
            "id": row["id"],
            "answer": 0,
            "scores": [],
            "image_path": "",
            "error": repr(e),
        })

    if (idx + 1) % 50 == 0:
        print(f"Done {idx+1}/{len(test_df)} | elapsed {(time.time()-t0)/60:.1f}m")

pred_df = pd.DataFrame(pred_rows)
submission = pred_df[["id", "answer"]].copy()
submission["answer"] = submission["answer"].astype(int)
submission.to_csv(SUB_PATH, index=False)
pred_df.to_csv(DEBUG_PATH, index=False)

print("Saved submission:", SUB_PATH)
print("Saved debug:", DEBUG_PATH)
print("Elapsed minutes:", (time.time() - t0) / 60)
display(submission.head())
display(submission["answer"].value_counts().sort_index())
display(pred_df[pred_df["error"] != ""].head())

Reloading best adapter for test inference...


/usr/local/lib/python3.12/dist-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


Done 50/1008 | elapsed 1.2m
Done 100/1008 | elapsed 2.2m
Done 150/1008 | elapsed 3.0m
Done 200/1008 | elapsed 3.9m
Done 250/1008 | elapsed 4.6m
Done 300/1008 | elapsed 5.3m
Done 350/1008 | elapsed 6.0m
Done 400/1008 | elapsed 6.7m
Done 450/1008 | elapsed 7.6m
Done 500/1008 | elapsed 8.5m
Done 550/1008 | elapsed 9.4m
Done 600/1008 | elapsed 10.3m
Done 650/1008 | elapsed 11.2m
Done 700/1008 | elapsed 12.0m
Done 750/1008 | elapsed 12.9m
Done 800/1008 | elapsed 13.8m
Done 850/1008 | elapsed 14.8m
Done 900/1008 | elapsed 15.7m
Done 950/1008 | elapsed 16.9m
Done 1000/1008 | elapsed 18.0m
Saved submission: /content/drive/MyDrive/pixels-to-predictions/submission_nli.csv
Saved debug: /content/drive/MyDrive/pixels-to-predictions/submission_nli_debug.csv
Elapsed minutes: 18.093719391028085


,id,answer
0,test_01750,2
1,test_00128,0
2,test_02891,0
3,test_02425,1
4,test_00930,0


,count
answer,
0,355
1,376
2,200
3,74
4,3


,id,answer,scores,image_path,error


In [ ]:
from google.colab import runtime
runtime.unassign()